# Model Forge — free AI 3D server on Google Colab

This notebook runs **your own** text-to-3D / image-to-3D engine (open-source Hunyuan3D-2) on Colab's free GPU and gives you a public URL to paste into the Model Forge app.

1. **Runtime → Change runtime type → T4 GPU** (or better).
2. Run the cells in order. Setup takes about 10–15 minutes the first time.
3. Copy the `https://….trycloudflare.com` URL from the last cell into the app: **Settings → My AI server URL**.

The free tier disconnects after a while of inactivity. When that happens, run the notebook again; you get a new URL each time.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%cd /content
# get the latest server code (updates it if you run this again)
!if [ -d 3dmodel ]; then git -C 3dmodel pull --ff-only; else git clone --depth 1 https://github.com/Dattaharshithreddy/3dmodel; fi
%cd /content/3dmodel/server
![ -d Hunyuan3D-2 ] || git clone --depth 1 https://github.com/Tencent/Hunyuan3D-2
!pip install -q -r Hunyuan3D-2/requirements.txt
!pip install -q -e Hunyuan3D-2
!pip install -q -r requirements.txt

### Optional: textures (recommended)
Compiles two small CUDA extensions (~5 min). Skip this cell to get untextured (grey) models faster.

In [ ]:
%cd /content/3dmodel/server/Hunyuan3D-2/hy3dgen/texgen/custom_rasterizer
!pip install -q --no-build-isolation .
%cd /content/3dmodel/server/Hunyuan3D-2/hy3dgen/texgen/differentiable_renderer
!pip install -q --no-build-isolation .
%cd /content/3dmodel/server

### Start the server
`LOW_VRAM=1` lets the free T4 (16 GB) run shapes and textures. Set `API_TOKEN` if you don't want strangers using your URL, and type the same token in the app's settings.

In [ ]:
import os, subprocess, time, re
os.environ['LOW_VRAM'] = '1'
os.environ['API_TOKEN'] = ''   # e.g. 'my-secret-123' (then enter the same in the app)
%cd /content/3dmodel/server
!pkill -f 'python server.py' || true
server = subprocess.Popen(['python', 'server.py'], stdout=open('server.log', 'w'), stderr=subprocess.STDOUT)
time.sleep(15)
!tail -5 server.log
!curl -s localhost:8000/health

In [ ]:
# Public URL via Cloudflare quick tunnel (free, no account needed)
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
tunnel = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://localhost:8000'], stdout=open('tunnel.log', 'w'), stderr=subprocess.STDOUT)
url = None
for _ in range(60):
    time.sleep(1)
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('tunnel.log').read())
    if m: url = m.group(0); break
print('\n=====  Paste this into the app (Settings → My AI server URL)  =====\n')
print(url or 'Tunnel not ready — check tunnel.log')

The first generation downloads the AI models (several GB), so it takes a few minutes. Later ones take about 1–3 minutes on a T4. To see what the server is doing:

In [ ]:
!tail -20 server.log